# NeutronToolKit — run without WSL or Docker

This notebook installs the public [NeutronToolKit](https://github.com/kbhalajiyadav/NeutronToolKit) repository into a temporary Google Colab Python runtime. It provides **reduced scattering curve plots, model previews, contrast-design scenarios, and optional file upload**. No Windows administrator access is needed.

**Scientific limits:** Example curves are synthetic. The deuteration score is a conditional *component-SLD-variance proxy*, not a validated SANS peak-height prediction or cost estimate. Do not upload unpublished, restricted ORNL, confidential VCU, or otherwise sensitive research files unless their cloud processing has been authorized.

## 1. Install and verify the current toolkit

**Run this cell first** (the play button on the code cell). Colab erases installed packages when its runtime resets, so repeat it after reconnecting to a new runtime.

This cell clones or updates the GitHub source, installs it into the same Python interpreter as the notebook, and immediately checks the package import. **Do not proceed until it prints Import success.** If it fails, inspect the error in this cell; Section 2 cannot recover a failed install.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kbhalajiyadav/NeutronToolKit.git"
ROOT = Path("/content/NeutronToolKit")

# Repeat after any runtime reset. It should finish with "Import success".
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
elif not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError(f"{ROOT} exists but is not a valid toolkit checkout")
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)

# Install into this notebook interpreter. Check pip's output if it fails.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(ROOT),
     "pytest", "pandas", "ipywidgets"],
    check=True,
)

# Make the repository src path explicit for runtimes with stale import paths.
SOURCE = str(ROOT / "src")
if SOURCE not in sys.path:
    sys.path.insert(0, SOURCE)
os.chdir(ROOT)

import neutrontoolkit
from neutrontoolkit.cli import doctor
git_sha = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
print("Import success:", neutrontoolkit.__version__, "from", neutrontoolkit.__file__)
print("Git commit:", git_sha)
print("Python:", sys.version.split()[0])
print("Scientific package check:", doctor()["packages"])

## 2. Check the scientific environment

This checks the installed package metadata; it does not certify a structural interpretation.

In [ ]:
import json
from pathlib import Path

if not (Path("/content/NeutronToolKit") / "pyproject.toml").exists():
    raise RuntimeError("Toolkit checkout absent. Run Section 1 first, in this runtime.")

try:
    from neutrontoolkit.cli import doctor
except ModuleNotFoundError as error:
    raise RuntimeError(
        "NeutronToolKit is not importable in the current session. "
        "Run Section 1 above; confirm the output says 'Import success'."
    ) from error

print(json.dumps(doctor(), indent=2))

## 3. Make your first reduced-data plot

The bundled curve is synthetic teaching data. The library saves PNG/SVG and provenance JSON; the notebook also displays the PNG inline.

In [ ]:
from IPython.display import Image, display
from neutrontoolkit.plots import plot_files

figure = plot_files(
    ["data/examples/synthetic_iq.dat"],
    kind="iq",
    outdir="outputs/colab/plots",
)
display(Image(filename=str(figure)))
print("Plot saved:", figure)

## 4. Preview a SasView model and its physical eligibility

The sphere preview is only a software example; it is **not a model proposed for a neat cholesteric phase**. The guide suggests testable hypotheses but does not identify a structure automatically.

In [ ]:
from neutrontoolkit.models import preview
from neutrontoolkit.model_rules import from_file

model_plot = preview(
    "config/example_sphere.json",
    outdir="outputs/colab/model",
)
display(Image(filename=str(model_plot)))
guide = from_file("config/model_guidance_cholesteric.json")
print("Hypotheses:", json.dumps(guide["candidates"], indent=2))
print("Scientific cautions:", *guide["cautions"], sep="\n • ")

## 5. Sweep nominal H/D label fractions

This explores site-specific H/D substitutions in example cholesterol-derived components. It uses **assumed densities** and has **no supplier pricing**. The score measures a conditional component-SLD-variance proxy; it cannot predict a 3 nm peak or decide a purchase.

In [ ]:
from neutrontoolkit.deuteration import write_sweep
import pandas as pd

summary_path = write_sweep(
    "config/deuteration_design_example.json",
    outdir="outputs/colab/deuteration",
)
report = json.loads(summary_path.read_text(encoding="utf-8"))
scenarios = pd.read_csv(summary_path.parent / "all_scenarios.csv")
display(pd.DataFrame([
    {
        "composition": row["composition"],
        "scenario_count": row["candidate_count"],
        "Pareto axis": row["pareto_resource"],
        "real dollar costs available": row["actual_cost_ranking_available"],
    }
    for row in report["summaries"]
]))
print("Saved:", summary_path.parent)
print("Important limitation:", report["limitations"][0])

### Explore a reference formulation

Plot the conditional contrast proxy against the number of deuterium atoms used per reference-batch mole balance. **This is not a cost-performance curve.**

In [ ]:
import matplotlib.pyplot as plt

subset = scenarios.loc[scenarios["composition"] == "reference_20_40_40"]
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.scatter(
    subset["D_atom_mol_per_reference_batch"],
    subset["component_variance_proxy"],
    s=8, alpha=0.45,
)
ax.set(
    xlabel="D-atom moles per reference batch (not monetary cost)",
    ylabel="Conditional component SLD variance proxy",
    title="Exploratory isotope-label screening — NOT predicted peak enhancement",
)
ax.grid(alpha=0.25)
plt.show()

## 6. Optional: upload reduced experimental ASCII files

Upload only files approved for cloud processing. This first-stage reader accepts `.dat`, `.txt`, `.csv`, and `.iq` with columns **Q, I(Q), optional dI, optional dQ** in verified compatible units. Colab storage is temporary. Do **not** upload raw NeXus data or restricted facility datasets.

In [ ]:
from google.colab import files as colab_files
from neutrontoolkit.files import EXTENSIONS, discover

uploaded = colab_files.upload()
target_dir = ROOT / "data" / "reduced"
target_dir.mkdir(parents=True, exist_ok=True)
for original_name, content in uploaded.items():
    name = Path(original_name).name
    if Path(name).suffix.lower() not in EXTENSIONS:
        print("Skipped unsupported file:", name)
        continue
    (target_dir / name).write_bytes(content)
    print("Saved:", name)
print("Reduced files:", len(discover(target_dir)))

## 7. Optional: filter and select files using widgets

This replaces the terminal **Space / P / U / /** interface in Colab. Select multiple file names, choose a plot mode, and click **Plot selected**. Your upload cell must run first if you want to use experimental files.

In [ ]:
from google.colab import output as colab_output
colab_output.enable_custom_widget_manager()
import ipywidgets as widgets
from IPython.display import clear_output

available = [
    str(path.relative_to(ROOT))
    for directory in ("data/examples", "data/reduced")
    for path in discover(ROOT / directory)
]
file_filter = widgets.Text(description="Filter:", placeholder="e.g. run or sample name")
selector = widgets.SelectMultiple(options=available, rows=9, description="Files:")
kind = widgets.Dropdown(
    options=[("I(Q)", "iq"), ("Kratky", "kratky"),
             ("Porod", "porod"), ("Guinier", "guinier")],
    value="iq", description="View:",
)
plot_button = widgets.Button(description="Plot selected", button_style="primary")
clear_button = widgets.Button(description="Unselect")
result_area = widgets.Output()

def update_filter(change):
    query = file_filter.value.casefold()
    selector.options = [p for p in available if query in p.casefold()]

def clear_selection(button):
    selector.value = ()

def run_plot(button):
    with result_area:
        clear_output(wait=True)
        if not selector.value:
            print("Select at least one reduced data file.")
            return
        try:
            path = plot_files(
                [ROOT / p for p in selector.value],
                kind=kind.value,
                outdir="outputs/colab/selected",
            )
            display(Image(filename=str(path)))
            print("Saved:", path)
        except (OSError, ValueError) as error:
            print("Data/plot validation error:", error)

file_filter.observe(update_filter, names="value")
plot_button.on_click(run_plot)
clear_button.on_click(clear_selection)
display(widgets.VBox([
    file_filter, selector, kind,
    widgets.HBox([plot_button, clear_button]),
    result_area,
]))

## 8. Optional: run automated tests and download outputs

Tests validate software functions and synthetic examples, not measured ORNL reduction or physical model selection. Download any useful plots and result tables before ending the Colab session.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], check=True)

In [ ]:
import shutil
from google.colab import files as colab_files
archive_path = shutil.make_archive("/content/neutrontoolkit-colab-outputs", "zip", "outputs/colab")
colab_files.download(archive_path)

## After Colab

Use [the WSL installation guide](https://github.com/kbhalajiyadav/NeutronToolKit/blob/main/docs/setup-fresh-laptop.md) when your lab computer's virtualization features are enabled. Colab is ideal for learning and small tests; it is not an ORNL data-reduction environment or substitute for a verified VCU Athena HPC job.